<a href="https://colab.research.google.com/github/Not-kh-lily-23/pulsar-conformal-triage/blob/main/stat_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import xml.etree.ElementTree as ET
readme_path='/content/README'
example_path='/content/example.phcx'

print("README (contents)")
try:
    with open(readme_path,'r') as f:
        print(f.read()[:1000])
except Exception as e:
    print("could not read README:",e)
print("XML structure of example.phcx")
try:
    tree=ET.parse(example_path)
    def print_tree(element,level=0):
        print("  "*level+f"- {element.tag}")
        seen_tags=set()
        for child in element:
            if child.tag not in seen_tags:
                print_tree(child,level+1)
                seen_tags.add(child.tag)
    print_tree(tree.getroot())
except Exception as e:
    print("could not parse XML:",e)

README (contents)
could not read README: [Errno 2] No such file or directory: '/content/README'
XML structure of example.phcx
- phcf
  - head
    - SourceID
    - Telescope
    - Coordinate
      - RA
      - Dec
      - Epoch
    - CentreFreq
    - BandWidth
    - MjdStart
    - ObservationLength
    - Origin
      - UTC
      - Beam
      - FileName
  - Section
    - BestValues
      - TopoPeriod
      - BaryPeriod
      - Dm
      - Accn
      - Jerk
      - Snr
      - Width
    - SampleRate
    - SubIntegrations
    - SubBands
    - Profile
    - SnrBlock
      - PeriodIndex
      - DmIndex
      - AccnIndex
      - JerkIndex
      - DataBlock


In [8]:
import xml.etree.ElementTree as ET
import numpy as np
from scipy.stats import skew, kurtosis
example_path='/content/example.phcx'
def hex_array(hex_str):
    clean_hex=''.join(hex_str.split())
    return np.frombuffer(bytes.fromhex(clean_hex),dtype=np.uint8).astype(float)
try:
    tree=ET.parse(example_path)
    root=tree.getroot()
    profile_node=root.find('.//Section/Profile')
    if profile_node is not None and profile_node.text:
        profile_array=hex_array(profile_node.text)
    else:
        raise ValueError("array not found")
    snr_node=root.find('.//Section/SnrBlock/DataBlock')
    if snr_node is not None and snr_node.text:
        snr_array=hex_array(snr_node.text)
    else:
        raise ValueError("dm-snr datablock not found")
    features={
        "mean_profile":np.mean(profile_array),
        "std_profile":np.std(profile_array),
        "kurtosis_profile":kurtosis(profile_array,fisher=True),
        "skewness_profile":skew(profile_array),
        "mean_dmsnr":np.mean(snr_array),
        "std_dmsnr":np.std(snr_array),
        "kurtosis_dmsnr":kurtosis(snr_array,fisher=True),
        "skewness_dmsnr":skew(snr_array)
    }
    print("features for dataset compatibility")
    for name,val in features.items():
        print(f"{name}: {val:.4f}")
except Exception as e:
    print(f"Feature extraction failed: {e}")

features for dataset compatibility
mean_profile: 16.4531
std_profile: 35.5853
kurtosis_profile: 31.7151
skewness_profile: 5.5610
mean_dmsnr: 53.9472
std_dmsnr: 38.8890
kurtosis_dmsnr: 3.9853
skewness_dmsnr: 1.7796
